In [3]:
import os
import re
import pandas as pd

# ------------------------------------------------------------
# Input directories in the exact order you want the output columns
# ------------------------------------------------------------
DIRECTORIES = [
    r"C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\Illumina_no_polisher\Pathogenwatch",
    r"C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\Nanopore_no_polisher_DONE\pathogenwatch",
    r"C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\nanopore_racon\kraken_contig_result\Pathogenwatch",
    r"C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\nanopore_medaka_DONE\Pathogenwatch",
    r"C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\nanopore_homopolisher_DONE\Pathogenwatch",
    r"C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\nanopore_racon_medaka\kraken_contig_result\Pathogenwatch",
    r"C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\nanopore_medaka_homopolisher_DONE\Pathogenwatch",
    r"C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\nanopore_medaka_polypolisher_pypolca\Pathogenwatch",
]

# Correct output column names in the same order as DIRECTORIES
COLUMN_NAMES = [
    "Illumina",
    "Nanopore",
    "Nanopore_racon",
    "Nanopore_medaka",
    "Nanopore_homopolisher",
    "Nanopore_racon_medaka",
    "Nanopore_medaka_homopolisher",
    "Hybrid",
]

INPUT_FILENAME = "ready_for_R_plot_final.csv"

# Output files
OUTPUT_DIR = r"C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\Plotting"
OUTPUT_PENICILLIN = os.path.join(OUTPUT_DIR, "penicillin_adjusted_matrix.csv")
OUTPUT_SEROTYPE = os.path.join(OUTPUT_DIR, "serotype_matrix.csv")
OUTPUT_MLST = os.path.join(OUTPUT_DIR, "mlst_matrix.csv")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def normalize_colname(name: str) -> str:
    name = str(name).replace("\n", " ").replace("\r", " ")
    name = re.sub(r"\s+", " ", name).strip().lower()
    return name

def build_col_lookup(columns):
    return {normalize_colname(c): c for c in columns}

def read_target_columns(csv_path):
    df = pd.read_csv(csv_path)
    lookup = build_col_lookup(df.columns)

    required = ["index", "penicillin_adjusted", "serotype", "mlst"]
    missing = [c for c in required if c not in lookup]
    if missing:
        raise ValueError(
            f"Missing required columns {missing} in {csv_path}\n"
            f"Available columns: {list(df.columns)}"
        )

    sub = df[
        [
            lookup["index"],
            lookup["penicillin_adjusted"],
            lookup["serotype"],
            lookup["mlst"],
        ]
    ].copy()

    sub.columns = ["Index", "Penicillin_adjusted", "Serotype", "mlst"]
    sub["Index"] = sub["Index"].astype(str).str.strip()
    sub = sub.drop_duplicates(subset=["Index"])

    return sub

def merge_one_field(directories, output_names, field_name):
    merged = None

    for folder, col_name in zip(directories, output_names):
        csv_path = os.path.join(folder, INPUT_FILENAME)

        if not os.path.exists(csv_path):
            print(f"Warning: file not found: {csv_path}")
            continue

        try:
            sub = read_target_columns(csv_path)[["Index", field_name]].copy()
        except Exception as e:
            print(f"Warning: could not process {csv_path}: {e}")
            continue

        sub.columns = ["Index", col_name]

        if merged is None:
            merged = sub
        else:
            merged = pd.merge(merged, sub, on="Index", how="outer")

    if merged is None:
        return pd.DataFrame(columns=["Index"] + output_names)

    merged = merged.sort_values("Index").reset_index(drop=True)
    return merged

# ------------------------------------------------------------
# Build the 3 output tables
# ------------------------------------------------------------
penicillin_df = merge_one_field(DIRECTORIES, COLUMN_NAMES, "Penicillin_adjusted")
serotype_df = merge_one_field(DIRECTORIES, COLUMN_NAMES, "Serotype")
mlst_df = merge_one_field(DIRECTORIES, COLUMN_NAMES, "mlst")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
penicillin_df.to_csv(OUTPUT_PENICILLIN, index=False)
serotype_df.to_csv(OUTPUT_SEROTYPE, index=False)
mlst_df.to_csv(OUTPUT_MLST, index=False)

print("Done.")
print(f"Penicillin file: {OUTPUT_PENICILLIN}")
print(f"Serotype file:   {OUTPUT_SEROTYPE}")
print(f"MLST file:       {OUTPUT_MLST}")

Done.
Penicillin file: C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\Plotting\penicillin_adjusted_matrix.csv
Serotype file:   C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\Plotting\serotype_matrix.csv
MLST file:       C:\Users\Amoeb\Documents\ZSW2025-2026\Waterloo_courses\BIOL499A\nanopore\Plotting\mlst_matrix.csv
